In [ ]:
import os
import pandas as pd
import numpy as np
import time 

kalman_filter_data_path = '../data/kaggle-drdataboston/attempt_2/updated_data_kalman_filtered'
matrix_path = '../data/kaggle-drdataboston/matrix.csv'

out_path = '../data/kaggle-drdataboston/attempt_2'

X = []
y_age = []
y_height = []
y_weight = []
y_gender = []

window_ms = 5000
interp_pts = 500

matrix = pd.read_csv(matrix_path)

start_time = time.perf_counter()

for idx, file in enumerate(os.listdir(kalman_filter_data_path)):
    if idx % 5 == 0:
        c_time = time.perf_counter()
        print(f"Processed {idx} files in {c_time - start_time:.6f} seconds")

    if not file.endswith('.csv'):
        continue

    file_path = os.path.join(kalman_filter_data_path, file)
    df = pd.read_csv(file_path)


    clean_name = file
    if(clean_name.endswith('.csv.csv')):
        clean_name = clean_name[:-4]
    
    person_id = None
    gender = None
    height = None
    weight = None
    age = None
    for idx, row in matrix.iterrows():
        if clean_name.strip().lower() in [str(row.iloc[i]).strip().lower() for i in range(1, 5)]:
            person_id = row.iloc[0]
            weight = row.iloc[5]
            age = row.iloc[6]
            height = row.iloc[7]
            gender = row.iloc[11]
            break

    if person_id is None:
        continue

    df['window_id'] = (df['timestamp'] - df['timestamp'].iloc[0]) // window_ms

    chunks = []
    labels_age = []
    labels_height = []
    labels_weight = []
    labels_gender = []

    for _, group in df.groupby('window_id'):
        if len(group) < 2:
            continue

        time_arr = group['timestamp'].values 
        mag = group['filtered'].values
        time_arr = time_arr - time_arr[0]

        new_time = np.linspace(0, time_arr[-1], interp_pts)
        interp_mag = np.interp(new_time, time_arr, mag)

        chunks.append(interp_mag)

        labels_age.append(age)
        labels_height.append(height)
        labels_weight.append(weight)
        labels_gender.append(gender)

    X.extend(chunks)
    y_age.extend(labels_age)
    y_height.extend(labels_height)
    y_weight.extend(labels_weight)
    y_gender.extend(labels_gender)


    print(np.array(X).shape)

X = np.array(X)
y_age = np.array(y_age)
y_height = np.array(y_height)
y_weight = np.array(y_weight)
y_gender = np.array(y_gender)

print(X.shape, y_age.shape, y_height.shape, y_weight.shape, y_gender.shape)
    

np.save(f"{out_path}/regression/timewise_5s_500p_X.npy", X)
np.save(f"{out_path}/regression/timewise_5s_500p_y_age.npy", y_age)
np.save(f"{out_path}/regression/timewise_5s_500p_y_height.npy", y_height)
np.save(f"{out_path}/regression/timewise_5s_500p_y_weight.npy", y_weight)
np.save(f"{out_path}/regression/timewise_5s_500p_y_gender.npy", y_gender)

